# EDA inicial — Rutas de Última Milla

**Objetivo de esta notebook:** cargar el dataset Amazon Last Mile Routing Research Challenge, aplanarlo a una tabla utilizable, y explorar su calidad antes de definir el modelo de riesgo de incumplimiento de SLA (Fase 1 de Insight).

**Decisión de diseño (ya tomada):** la tabla final tiene **una fila por parada (stop)**, no por ruta. El SLA se cumple o se rompe en cada entrega puntual, no en la ruta completa — si agregáramos a nivel ruta desde el arranque, perderíamos la posibilidad de identificar qué parada puntual fue la que incumplió.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

RAW = "../../data/raw/model_build_inputs"
PROCESSED = "../../data/processed"
FIGURES = "../../outputs/figures"

## 1. Carga de datos

Cargamos solo `route_data.json` y `package_data.json`. **No cargamos `travel_times.json`** a propósito: pesa 1.8 GB de los 2.2 GB totales (el resto del dataset pesa apenas ~400 MB), y es la matriz de tiempos de viaje entre paradas — sirve para optimizar secuencias de ruta (problema de TSP), no para el modelo de riesgo de SLA que estamos construyendo. Cargarlo acá sería usar memoria y tiempo en algo que no responde la pregunta de negocio.

In [ ]:
with open(f"{RAW}/route_data.json") as f:
    routes = json.load(f)

with open(f"{RAW}/package_data.json") as f:
    packages = json.load(f)

print(f"Rutas: {len(routes):,}")

## 2. Aplanar `route_data.json` a nivel parada

Cada ruta trae un diccionario `stops` con una entrada por parada (lat/lng/tipo/zona). Convertimos esto en una fila por `(route_id, stop_id)`, repitiendo los datos de la ruta (estación, fecha, `route_score`, etc.) en cada fila.

**Decisión de limpieza con impacto de negocio:** sacamos las paradas de tipo `Station` (el depósito de salida del camión). No es una entrega a un cliente, así que no tiene sentido de negocio incluirla en un análisis de cumplimiento de SLA — dejarla adentro inflaría el conteo de "paradas" sin aportar información real, y contaminaría cualquier métrica de cumplimiento con una parada que nunca tuvo una promesa de entrega.

In [ ]:
stop_rows = []
for route_id, r in routes.items():
    for stop_id, s in r["stops"].items():
        stop_rows.append({
            "route_id": route_id,
            "stop_id": stop_id,
            "station_code": r["station_code"],
            "date": r["date_YYYY_MM_DD"],
            "departure_time_utc": r["departure_time_utc"],
            "executor_capacity_cm3": r["executor_capacity_cm3"],
            "route_score": r["route_score"],
            "lat": s["lat"],
            "lng": s["lng"],
            "stop_type": s["type"],
            "zone_id": s["zone_id"],
        })

stops_df = pd.DataFrame(stop_rows)
print("Con Station incluida:", stops_df.shape)

stops_df = stops_df[stops_df["stop_type"] == "Dropoff"].copy()
print("Solo Dropoff (entregas reales):", stops_df.shape)

## 3. Agregar `package_data.json` a nivel parada

Una parada puede tener **más de un paquete** (34.1% de las paradas, verificado sobre el dataset completo). Como decidimos que la unidad de análisis es la parada, no el paquete, tenemos que **agregar** la info de todos los paquetes de una parada en una sola fila:

- `n_packages`: cantidad de paquetes en la parada.
- `has_time_window`: si **al menos un** paquete de la parada tiene ventana horaria especificada.
- `window_start_utc` / `window_end_utc`: la ventana más amplia entre todos los paquetes de la parada (min de los inicios, max de los finales).
- `total_volume_cm3`: volumen total (suma de depth×height×width de cada paquete).
- `total_planned_service_seconds`: tiempo de servicio total estimado en la parada.
- `any_rejected` / `any_attempted`: si algún paquete de la parada fue rechazado o tuvo un intento de entrega fallido.

**Decisión de limpieza con impacto de negocio (la más importante de esta notebook):** cuando un paquete no tiene ventana horaria (`start_time_utc`/`end_time_utc` en `NaN`), **no lo tratamos como dato faltante para descartar o imputar** — lo tratamos como una categoría real de negocio: "esta entrega no tenía una ventana horaria prometida al cliente". Imputar una ventana horaria inventada sería fabricar una promesa de SLA que nunca existió. Vamos a cuantificar cuántas paradas caen en esta categoría en la sección de gráficos, porque el resultado cambia por completo cómo se puede definir "incumplimiento de SLA" más adelante.

In [ ]:
pkg_rows = []
for route_id, stops in packages.items():
    for stop_id, pkgs in stops.items():
        starts, ends, vols, services = [], [], [], []
        any_rejected = False
        any_attempted = False
        for pkg_id, info in pkgs.items():
            tw = info.get("time_window", {})
            st, en = tw.get("start_time_utc"), tw.get("end_time_utc")
            if isinstance(st, str):
                starts.append(st)
            if isinstance(en, str):
                ends.append(en)
            dims = info.get("dimensions", {})
            d, h, w = dims.get("depth_cm", 0), dims.get("height_cm", 0), dims.get("width_cm", 0)
            vols.append((d or 0) * (h or 0) * (w or 0))
            services.append(info.get("planned_service_time_seconds", 0) or 0)
            if info.get("scan_status") == "REJECTED":
                any_rejected = True
            if info.get("scan_status") == "DELIVERY_ATTEMPTED":
                any_attempted = True
        pkg_rows.append({
            "route_id": route_id,
            "stop_id": stop_id,
            "n_packages": len(pkgs),
            "has_time_window": len(starts) > 0,
            "window_start_utc": min(starts) if starts else None,
            "window_end_utc": max(ends) if ends else None,
            "total_volume_cm3": sum(vols),
            "total_planned_service_seconds": sum(services),
            "any_rejected": any_rejected,
            "any_attempted": any_attempted,
        })

pkg_df = pd.DataFrame(pkg_rows)
print("pkg_df:", pkg_df.shape)

## 4. Merge final

In [ ]:
final_df = stops_df.merge(pkg_df, on=["route_id", "stop_id"], how="left")
print("Tabla final:", final_df.shape)
final_df.head()

## 5. Chequeos de calidad: duplicados, nulos, tipos

Antes de confiar en cualquier gráfico o número, verificamos que no haya sorpresas estructurales:

- **Duplicados**: la clave real de una parada es `(route_id, stop_id)`, no `stop_id` solo — la documentación del dataset advierte explícitamente que los IDs de parada se reutilizan entre rutas distintas (`AA`, `AB`, etc. no son únicos globalmente). Si chequeáramos duplicados solo por `stop_id` obtendríamos falsos positivos por todos lados.
- **Nulos**: esperamos nulos en `window_start_utc`/`window_end_utc` (ya sabemos que son ~93%, es una categoría de negocio, no un error) y podemos encontrar algunos en `zone_id`.

In [ ]:
print("Duplicados por (route_id, stop_id):", final_df.duplicated(subset=["route_id", "stop_id"]).sum())
print()
print("Nulos por columna:")
print(final_df.isna().sum())
print()
print("Tipos de datos:")
print(final_df.dtypes)

## 6. Guardar datos procesados

Guardamos en formato Parquet (no CSV): preserva tipos de datos (booleanos, fechas) sin que se rompan al releerlo, y pesa bastante menos que un CSV equivalente para este volumen de filas.

In [ ]:
final_df.to_parquet(f"{PROCESSED}/stops.parquet", index=False)
print("Guardado en data/processed/stops.parquet")

## 7. Gráficos exploratorios (3)

### Gráfico 1 — Distribución de `route_score`

`route_score` es el único indicador de calidad disponible para el 100% de las rutas (a diferencia de la ventana horaria, que falta en la mayoría). Importa ver su balance de clases desde ya, porque probablemente termine siendo la base del target de riesgo de SLA en F1-03 — y está muy desbalanceado hacia `Medium`/`High`, con muy pocos casos `Low`.

In [ ]:
order = ["Low", "Medium", "High"]
colors = ["#de350b", "#ff991f", "#00875a"]
counts = final_df["route_score"].value_counts().reindex(order)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(order, counts.values, color=colors)
for i, v in enumerate(counts.values):
    ax.text(i, v + counts.max()*0.02, f"{v:,}\n({v/len(final_df)*100:.1f}%)", ha="center", fontsize=9)
ax.set_title("Distribución de route_score a nivel parada")
ax.set_ylabel("Cantidad de paradas")
ax.set_ylim(0, counts.max()*1.2)
plt.tight_layout()
plt.savefig(f"{FIGURES}/fig1_route_score.png")
plt.show()

### Gráfico 2 — Paradas con vs. sin ventana horaria

Este es el hallazgo más importante del EDA: la gran mayoría de las paradas **no tienen ninguna ventana horaria prometida al cliente**. Esto no es un problema de datos faltantes que haya que "arreglar" — es información real sobre cómo opera Amazon en este dataset, y redefine qué significa "incumplimiento de SLA" para la Fase 1.

In [ ]:
vc = final_df["has_time_window"].value_counts()
labels = ["Sin ventana\nhoraria", "Con ventana\nhoraria"]
values = [vc.get(False, 0), vc.get(True, 0)]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values, color=["#5e6c84", "#0052cc"])
for b, v in zip(bars, values):
    ax.text(b.get_x() + b.get_width()/2, v + max(values)*0.02, f"{v:,}\n({v/len(final_df)*100:.1f}%)", ha="center", fontsize=9)
ax.set_title("Paradas con ventana horaria especificada")
ax.set_ylabel("Cantidad de paradas")
plt.tight_layout()
plt.savefig(f"{FIGURES}/fig2_ventana_horaria.png")
plt.show()

### Gráfico 3 — Paquetes por parada

Confirma que la mayoría de las paradas tienen un solo paquete, pero hay una cola larga de paradas con múltiples paquetes (hasta 113 en un caso). Justifica por qué agregamos package_data por parada en vez de dejarlo a nivel paquete.

In [ ]:
capped = final_df["n_packages"].clip(upper=10)
vc2 = capped.value_counts().reindex(range(1, 11), fill_value=0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, 11), vc2.values, color="#6554c0")
ax.set_xticks(range(1, 11))
ax.set_xticklabels([str(i) for i in range(1, 10)] + ["10+"])
ax.set_title("Paquetes por parada (10+ agrupado)")
ax.set_xlabel("Cantidad de paquetes en la parada")
ax.set_ylabel("Cantidad de paradas")
plt.tight_layout()
plt.savefig(f"{FIGURES}/fig3_paquetes_por_parada.png")
plt.show()

## 8. Conclusión de negocio

**En una línea:** el 93.4% de las paradas de este dataset no tienen una ventana horaria de entrega prometida al cliente, por lo que el riesgo de incumplimiento de SLA de la Fase 1 no se puede definir como "llegó tarde a la ventana horaria" para la mayoría de los casos — se construye en su lugar a partir de `route_score` (el indicador de calidad que Amazon calcula para el 100% de las rutas).

## 9. Definición de la variable objetivo (F1-03)

**Decisión:** el target del modelo de riesgo de SLA es `route_score` (`Low` / `Medium` / `High`), calculado por Amazon a nivel ruta y propagado a cada parada de esa ruta como etiqueta débil (*weak label*). Ya está disponible en `final_df` / `stops.parquet` — no requiere código adicional, se unió en la sección 2.

**Por qué a nivel parada y no a nivel ruta:** con ~898K paradas en vez de ~6K rutas, el modelo cuenta con muchísimo más volumen para aprender patrones reales, y el resultado es accionable a nivel operativo: permite identificar qué paradas puntuales son de riesgo dentro de una ruta, en lugar de solo poder marcar la ruta completa como problemática sin poder decir por qué.

**Limitación conocida, aceptada explícitamente:** al propagar el score de una ruta a cada una de sus paradas, no todas contribuyeron por igual a ese resultado — es una etiqueta con ruido (weak label). Se acepta este trade-off porque el ruido no está correlacionado con las variables predictoras y se diluye con el volumen de datos: la diferencia real entre grupos (por ejemplo, zonas con más incidencia de rutas `Low`) sigue siendo detectable aunque una porción de las etiquetas individuales sea imprecisa.

**Pendiente para la siguiente etapa (feature engineering / modelo baseline):** la clase `Low` representa solo 1.7% de las paradas — un desbalance de clases severo que va a condicionar las métricas de evaluación (accuracy no sirve con este desbalance) y posiblemente requiera balanceo o ponderación de clases.